# 📊 KoCulture-Dialogues 탐색적 데이터 분석 (EDA)

학습에 들어가기 전에 데이터의 특성을 직접 확인합니다. 이 분석은 README **II장**에 들어가며,
특히 `max_seq_length=512` 같은 하이퍼파라미터 선택의 **근거**가 됩니다.

**분석 항목**
1. 기본 통계 (총 행 수, 고유 신조어 수)
2. 신조어별 예시 수 분포 → `images/eda_examples_per_slang.png`
3. 토큰 길이 분포 → `images/eda_token_length.png` (→ `max_seq_length` 근거)


In [ ]:
# (Colab) 필요한 라이브러리 설치
!pip install -q -U datasets==3.0.0 transformers==4.46.0 matplotlib

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_dataset
from datasets.builder import VerificationMode
from transformers import AutoTokenizer

os.makedirs('images', exist_ok=True)
MODEL_ID = 'Qwen/Qwen2.5-3B-Instruct'

# 한글 그래프 폰트 (Colab). 설치 후 런타임 재시작이 필요할 수 있음
try:
    import matplotlib.font_manager as fm
    for cand in ['NanumGothic', 'Noto Sans CJK KR', 'Malgun Gothic', 'AppleGothic']:
        if any(cand in f.name for f in fm.fontManager.ttflist):
            plt.rcParams['font.family'] = cand
            break
    plt.rcParams['axes.unicode_minus'] = False
except Exception as e:
    print('폰트 설정 건너뜀:', e)

## 1. 데이터 로드 & 기본 통계

In [ ]:
ds_raw = load_dataset(
    'huggingface-KREW/KoCulture-Dialogues',
    split='train',
    verification_mode=VerificationMode.NO_CHECKS,
)

df = ds_raw.to_pandas()
print(f'총 행 수      : {len(df):,}')
print(f'고유 신조어 수 : {df["title"].nunique():,}')
print(f'필드          : {list(df.columns)}')
df.head(3)

## 2. 신조어별 예시 수 분포

평균만 보면 약 29개지만, 분포가 균등한지 롱테일인지가 학습 난이도에 영향을 줍니다.

In [ ]:
counts = df['title'].value_counts()

print('=== 신조어별 예시 수 요약 ===')
print(f'평균   : {counts.mean():.1f}')
print(f'중앙값 : {counts.median():.0f}')
print(f'최대   : {counts.max()}  ({counts.idxmax()})')
print(f'최소   : {counts.min()}  ({counts.idxmin()})')

print('\n=== 예시 많은 신조어 TOP 10 ===')
print(counts.head(10).to_string())
print('\n=== 예시 적은 신조어 BOTTOM 10 ===')
print(counts.tail(10).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(counts.values, bins=40, color='#2563EB', edgecolor='white')
ax.axvline(counts.mean(), color='#D97706', linestyle='--', linewidth=2,
           label=f'평균 {counts.mean():.1f}')
ax.set_xlabel('신조어당 예시 수')
ax.set_ylabel('신조어 개수')
ax.set_title('신조어별 예시 수 분포')
ax.legend()
fig.tight_layout()
fig.savefig('images/eda_examples_per_slang.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ 저장: images/eda_examples_per_slang.png')

## 3. 토큰 길이 분포 → `max_seq_length` 근거

각 대화(user + assistant)를 Qwen2.5 토크나이저로 인코딩한 길이를 측정합니다.
대부분의 샘플이 512 토큰 이내라면 `max_seq_length=512`가 충분하다는 근거가 됩니다.

In [ ]:
tok = AutoTokenizer.from_pretrained(MODEL_ID)

def chat_token_len(q, a):
    messages = [
        {'role': 'user', 'content': q},
        {'role': 'assistant', 'content': a},
    ]
    text = tok.apply_chat_template(messages, tokenize=False)
    return len(tok(text, add_special_tokens=False)['input_ids'])

lengths = np.array([chat_token_len(q, a) for q, a in zip(df['question'], df['answer'])])

print('=== 토큰 길이 요약 ===')
print(f'평균   : {lengths.mean():.1f}')
print(f'중앙값 : {np.median(lengths):.0f}')
print(f'95%    : {np.percentile(lengths, 95):.0f}')
print(f'99%    : {np.percentile(lengths, 99):.0f}')
print(f'최대   : {lengths.max()}')

within_512 = (lengths <= 512).mean() * 100
print(f'\n→ 512 토큰 이내 샘플 비율: {within_512:.2f}%')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(lengths, bins=50, color='#2563EB', edgecolor='white')
ax.axvline(512, color='#DC2626', linestyle='--', linewidth=2, label='max_seq_length = 512')
ax.axvline(np.percentile(lengths, 99), color='#D97706', linestyle=':', linewidth=2,
           label=f'99% = {np.percentile(lengths, 99):.0f}')
ax.set_xlabel('토큰 길이 (user + assistant)')
ax.set_ylabel('샘플 수')
ax.set_title('대화 토큰 길이 분포')
ax.legend()
fig.tight_layout()
fig.savefig('images/eda_token_length.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ 저장: images/eda_token_length.png')

## 4. 정리

위에서 출력된 실제 수치를 README II장의 🔴 표시 부분에 채워 넣으세요.

- 신조어별 예시 수: 최다 / 최소 / 중앙값
- 토큰 길이: 99% 분위수, 512 이내 비율 → `max_seq_length=512` 정당화
- 생성된 그래프 2개(`images/eda_examples_per_slang.png`, `images/eda_token_length.png`)를 repo에 커밋